In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("spam_train_cleaned.csv")
test_df = pd.read_csv("spam_test_cleaned.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

Train shape: (31311, 16)
Test shape : (7828, 16)

Train columns:
['label', 'urls', 'hour', 'combined_text', 'capital_letter_count', 'capital_ratio', 'exclamation_count', 'question_count', 'special_char_count', 'day_of_week_Friday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']


,label,urls,hour,combined_text,capital_letter_count,capital_ratio,exclamation_count,question_count,special_char_count,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,0,1,2,volunteers are needed for alumni phonothon 200...,37,0.055306,0,0,2,0,0,0,0,0,0,1
1,0,0,0,re opensuse opensuse and faxes on sunday 10 fe...,85,0.048935,0,2,42,0,0,0,0,0,0,1
2,0,1,2,re r matching a period in grep on 08 05 2008 0...,69,0.045128,0,4,114,0,0,0,0,0,0,1
3,1,1,17,fast and safe male enhancement huge love gun i...,11,0.034700,3,0,0,0,0,0,0,1,0,0
4,0,1,3,re python dev documentation reorganization was...,44,0.026113,0,0,94,0,0,0,0,0,0,1


In [ ]:
#Check duplicates
print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum())

Train duplicates: 3900
Test duplicates: 793


In [3]:
# Remove duplicate emails from train and test
train_df = train_df.drop_duplicates(subset="combined_text").reset_index(drop=True)
test_df = test_df.drop_duplicates(subset="combined_text").reset_index(drop=True)

print("Train shape after removing duplicates:", train_df.shape)
print("Test shape after removing duplicates:", test_df.shape)

print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum())

Train shape after removing duplicates: (27411, 16)
Test shape after removing duplicates: (7035, 16)
Train duplicates: 0
Test duplicates: 0


In [4]:
# check class balance 
print("Train class distribution:")
print(train_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())

Train class distribution:
label
0    13790
1    13621
Name: count, dtype: int64

Test class distribution:
label
1    3580
0    3455
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix


# X and y


X = train_df.drop(columns=["label"])
y = train_df["label"]

X_test_raw = test_df.drop(columns=["label"])
y_test = test_df["label"]



# Train / Validation split


X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



# Text + numerical columns


TEXT_COL = "combined_text"

NUMERIC_COLS = [
    col for col in X.columns
    if col != TEXT_COL
]



# TF-IDF


tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(X_train_raw[TEXT_COL])
X_text_val = tfidf.transform(X_val_raw[TEXT_COL])
X_text_test = tfidf.transform(X_test_raw[TEXT_COL])



# Scaling


scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_train_raw[NUMERIC_COLS])
X_num_val = scaler.transform(X_val_raw[NUMERIC_COLS])
X_num_test = scaler.transform(X_test_raw[NUMERIC_COLS])



# Combine ALL features


X_train = hstack([
    X_text_train,
    csr_matrix(X_num_train)
])

X_val = hstack([
    X_text_val,
    csr_matrix(X_num_val)
])

X_test = hstack([
    X_text_test,
    csr_matrix(X_num_test)
])


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (21928, 314)
Validation: (5483, 314)
Test: (7035, 314)


In [6]:
#Default SVM no hyperpramters
from sklearn.svm import SVC

svm_default = SVC()

svm_default.fit(X_train, y_train)

print("Default SVM trained!")

Default SVM trained!


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_train_pred = svm_default.predict(X_train)
y_val_pred = svm_default.predict(X_val)

print("========== DEFAULT SVM ==========")

print("\nTRAINING PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_train, y_train_pred), 4))
print("Precision:", round(precision_score(y_train, y_train_pred), 4))
print("Recall   :", round(recall_score(y_train, y_train_pred), 4))
print("F1 Score :", round(f1_score(y_train, y_train_pred), 4))

print("\nVALIDATION PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_pred), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred), 4))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_val_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_val, y_val_pred))

========== DEFAULT SVM ==========

TRAINING PERFORMANCE
Accuracy : 0.9859
Precision: 0.9823
Recall   : 0.9894
F1 Score : 0.9858

VALIDATION PERFORMANCE
Accuracy : 0.9808
Precision: 0.9774
Recall   : 0.9842
F1 Score : 0.9808

CONFUSION MATRIX
[[2696   62]
 [  43 2682]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2758
           1       0.98      0.98      0.98      2725

    accuracy                           0.98      5483
   macro avg       0.98      0.98      0.98      5483
weighted avg       0.98      0.98      0.98      5483



- note: Difference: 0.51 percentage points

That's a very small train–validation gap, which is actually excellent evidence that the SVM is generalizing well.

In [8]:
#Hyperpramaters C
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

C_values = [0.01, 0.1, 1, 10, 100]

C_results = []

for C in C_values:

    model = SVC(
        C=C,
        kernel="rbf",
        gamma="scale"
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    C_results.append({
        "C": C,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Validation Accuracy": accuracy_score(y_val, val_pred),
        "Train F1": f1_score(y_train, train_pred),
        "Validation F1": f1_score(y_val, val_pred),
        "F1 Gap": (
            f1_score(y_train, train_pred)
            - f1_score(y_val, val_pred)
        )
    })

C_results_df = pd.DataFrame(C_results)

display(C_results_df)



,C,Train Accuracy,Validation Accuracy,Train F1,Validation F1,F1 Gap
0,0.01,0.881567,0.880175,0.891389,0.890409,0.000980
1,0.10,0.965888,0.962065,0.966111,0.962373,0.003738
2,1.00,0.985863,0.980850,0.985825,0.980801,0.005024
3,10.00,0.996124,0.985592,0.996101,0.985544,0.010556
4,100.00,0.999863,0.986504,0.999862,0.986472,0.013391


I would choose C = 1

It gives you:

Very high validation F1: 98.08%
Very high validation accuracy: 98.09%
Small train-validation gap: 0.50%
Much less overfitting than C=10 or C=100

C=100 technically has the highest validation F1 (98.65%), but look at the training F1:

99.986% training vs 98.647% validation

That's a 1.34 percentage-point gap.

And C=10 has a similar issue.

What C is doing here

As C increases, SVM becomes more focused on correctly classifying the training examples, allowing fewer training errors.

In [9]:

from sklearn.svm import SVC

svm_tuned = SVC(
    C=1,
    kernel="rbf",
    gamma="scale"
)

svm_tuned.fit(X_train, y_train)

y_train_pred_tuned = svm_tuned.predict(X_train)
y_val_pred_tuned = svm_tuned.predict(X_val)

I tested different C values. As C increased, the model became more focused on minimizing training errors. C=100 achieved almost 100% training F1, but its validation performance did not increase by much, creating a larger train-validation gap. C=1 gave us a 98.58% training F1 and 98.08% validation F1, with only a 0.50% gap. Therefore, C=1 provides the best balance between high performance and generalization, with little evidence of overfitting.